In [2]:
"""
Created on: 18-08-2026

@author B.A. Sturre

PyTorch tutorial 
https://www.youtube.com/watch?v=c36lUUr864M 
"""

import matplotlib.pyplot as plt
import numpy as np
from sklearn import datasets 
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn

## Logistic Regression

In [14]:
# binary classification problem 
bc = datasets.load_breast_cancer()
x,y = bc.data, bc.target
n_samples, n_features = x.shape
#print(n_samples, n_features)

# splitting in training and testing data
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=1234)

# scaling features to 0 mean and unit variance
# reccomended to do when logistic regression. 
sc = StandardScaler()
x_train = sc.fit_transform(x_train)
x_test = sc.transform(x_test)

# making them torch arrays
x_train = torch.from_numpy(x_train.astype(np.float32))
x_test = torch.from_numpy(x_test.astype(np.float32))
y_train = torch.from_numpy(y_train.astype(np.float32))
y_test = torch.from_numpy(y_test.astype(np.float32))

# reshape y tensors
y_train = y_train.view(y_train.shape[0], 1)
y_test = y_test.view(y_test.shape[0], 1)

# define model 
# f = wx + b, sigmoid at the end
class LogisticRegression(nn.Module): 

    def __init__(self, n_input_features): 
        super(LogisticRegression, self).__init__()
        self.linear = nn.Linear(n_input_features, 1)

    def forward(self, x):
        y_pred = torch.sigmoid(self.linear(x))
        return y_pred 
    

input_size = n_features
output_size = 1
model = LogisticRegression(input_size)

# loss and optimizer
learning_rate = 0.01
criterion = nn.BCELoss()  # binary cross entropy loss
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

# training loop 
num_epochs = 100 
for epoch in range(num_epochs): 
    # forward pass 
    y_pred = model(x_train)

    # loss 
    loss = criterion(y_pred, y_train)

    # backward pass 
    loss.backward()

    # update
    optimizer.step()

    optimizer.zero_grad()

    if epoch % 10 == 9: 
        print(f'epoch {epoch+1}: loss = {loss.item():.4f}')

# evaluate model 
with torch.no_grad():
    y_predicted = model(x_test)
    y_predicted_cls = y_predicted.round()  # sigmoid will return value between 0 and 1, if larger than 0.5 then 1 and otherwise 0

    # calculating accuracy 
    acc = y_predicted_cls.eq(y_test).sum()  # adds 1 for every prediction that is correct
    acc = acc / float(y_test.shape[0])  # divide by number of test samples 
    print(f'accuracy = {acc:.4f}')

epoch 10: loss = 0.5340
epoch 20: loss = 0.4483
epoch 30: loss = 0.3921
epoch 40: loss = 0.3523
epoch 50: loss = 0.3222
epoch 60: loss = 0.2986
epoch 70: loss = 0.2795
epoch 80: loss = 0.2636
epoch 90: loss = 0.2501
epoch 100: loss = 0.2385
accuracy = 0.9211
